In [0]:
print(spark.version)
spark.sql("SHOW CATALOGS").show()

4.2.0
+------------+
|     catalog|
+------------+
|datamodeling|
|   dbacademy|
|     samples|
|      system|
|   workspace|
+------------+



#Creating the three schemas for the medallion architecture

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;
CREATE SCHEMA IF NOT EXISTS workspace.silver;
CREATE SCHEMA IF NOT EXISTS workspace.gold;

bronze Layer

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw_files;

In [0]:
%sql
SHOW SCHEMAS IN workspace;
SHOW VOLUMES IN workspace.bronze;

database,volume_name
bronze,raw_files


##Importing data files

In [0]:
%pip install gdown

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
import requests
try:
    r = requests.get("https://drive.google.com", timeout=10)
    print("Reachable:", r.status_code)
except Exception as e:
    print("Blocked:", e)

Reachable: 200


In [0]:
import gdown
from pathlib import Path

folder_url = "https://drive.google.com/drive/folders/1Rjz_L4Depe55UA_uRoOGkXar-c7gHk_C?usp=drive_link"

output_dir = Path("/Volumes/workspace/bronze/raw_files")

files_exist = any(output_dir.glob("Urban_*"))

if files_exist:
    print("Files already exist.")
else:
    gdown.download_folder(
        url=folder_url,
        output="/Volumes/workspace/bronze/raw_files",
        quiet=False,
        use_cookies=False
    )

Files already exist.


In [0]:
%fs ls /Volumes/workspace/bronze/raw_files

path,name,size,modificationTime
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-04.csv,429662745,1789485310000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-05.csv,492535780,1789485320000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csv,462646895,1789285640000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-06.csvo317dudq.part,Urban_Flow_Analytics_Taxi_Dataset_2025-06.csvo317dudq.part,23068672,1789485324000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-07.csv,418006505,1789285649000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,383928038,1789285658000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-09.csv,456686968,1789285668000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-10.csv,477118702,1789285679000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-11.csv,449095149,1789285689000
dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,Urban_Flow_Analytics_Taxi_Dataset_2025-12.csv,460962917,1789285700000


In [0]:
from pyspark.sql.functions import lit, col, regexp_extract

df = (
    spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("/Volumes/workspace/bronze/raw_files/*.csv")
    .withColumn("source_file", col("_metadata.file_path"))
    .withColumn("source_month", regexp_extract("source_file", r"(\d{4}-\d{2})\.csv", 1))
)

df.printSchema()
df.count()


root
 |-- provider_code: integer (nullable = true)
 |-- pickup_timestamp: timestamp (nullable = true)
 |-- dropoff_timestamp: timestamp (nullable = true)
 |-- rider_count: double (nullable = true)
 |-- distance_miles: double (nullable = true)
 |-- rate_class_id: double (nullable = true)
 |-- offline_record_flag: string (nullable = true)
 |-- origin_loc_id: integer (nullable = true)
 |-- dest_loc_id: integer (nullable = true)
 |-- fare_settlement_method: integer (nullable = true)
 |-- base_fare: double (nullable = true)
 |-- surcharge_misc: double (nullable = true)
 |-- transit_tax: double (nullable = true)
 |-- driver_tip_payment: double (nullable = true)
 |-- toll_total: double (nullable = true)
 |-- service_improvement_fee: double (nullable = true)
 |-- charge_total: double (nullable = true)
 |-- zone_congestion_fee: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- congestion_relief_fee: double (nullable = true)
 |-- source_file: string (nullable = false)
 |--

48601782

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("workspace.bronze.trips_raw")
)

In [0]:
%sql
SELECT * FROM bronze.trips_raw LIMIT 5;

provider_code,pickup_timestamp,dropoff_timestamp,rider_count,distance_miles,rate_class_id,offline_record_flag,origin_loc_id,dest_loc_id,fare_settlement_method,base_fare,surcharge_misc,transit_tax,driver_tip_payment,toll_total,service_improvement_fee,charge_total,zone_congestion_fee,Airport_fee,congestion_relief_fee,source_file,source_month
1,2025-08-28T20:32:15.000Z,2025-08-28T20:37:40.000Z,2.0,1.1,1.0,N,170,234,1,7.2,4.25,0.5,2.6,0.0,1.0,15.55,2.5,0.0,0.75,dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,2025-08
1,2025-08-28T20:49:33.000Z,2025-08-28T21:15:10.000Z,1.0,5.9,1.0,N,231,238,1,27.5,4.25,0.5,10.0,0.0,1.0,43.25,2.5,0.0,0.75,dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,2025-08
1,2025-08-28T20:29:55.000Z,2025-08-28T20:36:33.000Z,1.0,0.6,1.0,N,230,230,1,7.9,4.25,0.5,4.05,0.0,1.0,17.7,2.5,0.0,0.75,dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,2025-08
1,2025-08-28T20:53:36.000Z,2025-08-28T20:54:08.000Z,1.0,0.0,5.0,N,230,230,3,0.01,0.75,0.0,0.0,0.0,1.0,1.76,0.0,0.0,0.75,dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,2025-08
1,2025-08-28T20:55:19.000Z,2025-08-28T20:55:59.000Z,0.0,0.0,5.0,N,230,230,1,16.0,0.75,0.0,0.0,0.0,1.0,17.75,0.0,0.0,0.75,dbfs:/Volumes/workspace/bronze/raw_files/Urban_Flow_Analytics_Taxi_Dataset_2025-08.csv,2025-08


In [0]:
zone_lookup = spark.read.csv(
    "/Volumes/workspace/bronze/raw_files/reference/Urban_Flow_Analytics_Zone_Dataset.csv",
    header=True, inferSchema=True
)

###Sanity check on the reference dataset 

In [0]:
zone_lookup.head()

Row(loc_id=1, borough_name='EWR', zone_name='Newark Airport', service_zone='EWR')

In [0]:
zone_lookup.count()

265

In [0]:
zone_lookup.select("borough_name").distinct().show()

+-------------+
| borough_name|
+-------------+
|          EWR|
|          N/A|
|Staten Island|
|      Unknown|
|        Bronx|
|     Brooklyn|
|    Manhattan|
|       Queens|
+-------------+



###Geocoding

In [0]:
%pip install geopandas shapely

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%pip install fsspec requests

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
%restart_python

In [0]:
import geopandas as gpd
import requests
import tempfile

# Download the parquet file first
url = "https://data.source.coop/cholmes/nyc-taxi-zones/taxi_zones_4326.parquet"
response = requests.get(url, timeout=30)
temp_path = "/tmp/taxi_zones_4326.parquet"
with open(temp_path, 'wb') as f:
    f.write(response.content)


In [0]:
# Read from local file
zones_gdf = gpd.read_parquet(temp_path)
zones_gdf["centroid_lat"] = zones_gdf.geometry.centroid.y
zones_gdf["centroid_lon"] = zones_gdf.geometry.centroid.x
zones_geo = zones_gdf[["LocationID", "centroid_lat", "centroid_lon"]]

/home/spark-38262580-b1d8-4bd3-acb3-07/.ipykernel/74/command-7518030493947764-1233586534:3: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zones_gdf["centroid_lat"] = zones_gdf.geometry.centroid.y
/home/spark-38262580-b1d8-4bd3-acb3-07/.ipykernel/74/command-7518030493947764-1233586534:4: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  zones_gdf["centroid_lon"] = zones_gdf.geometry.centroid.x


In [0]:
zones_geo.tail()

,LocationID,centroid_lat,centroid_lon
258,259,40.897932,-73.852215
259,260,40.744234,-73.906306
260,261,40.709138,-74.013023
261,262,40.775932,-73.946510
262,263,40.778765,-73.951010


In [0]:
zone_lookup = spark.read.csv(
    "/Volumes/workspace/bronze/raw_files/reference/Urban_Flow_Analytics_Zone_Dataset.csv",
    header=True, inferSchema=True
)
zone_geo_spark = spark.createDataFrame(zones_geo).withColumnRenamed("LocationID", "loc_id")

dim_zone = zone_lookup.join(zone_geo_spark, on="loc_id", how="left")
dim_zone.write.format("delta").mode("overwrite").saveAsTable("workspace.silver.dim_zone")

In [0]:
%sql
SELECT * FROM silver.dim_zone LIMIT 10;

loc_id,borough_name,zone_name,service_zone,centroid_lat,centroid_lon
1,EWR,Newark Airport,EWR,40.69183115473482,-74.17400008189793
2,Queens,Jamaica Bay,Boro Zone,40.61674466444515,-73.83129879543229
3,Bronx,Allerton/Pelham Gardens,Boro Zone,40.86447314470219,-73.84742222112844
4,Manhattan,Alphabet City,Yellow Zone,40.723751609058645,-73.97696808881015
5,Staten Island,Arden Heights,Boro Zone,40.55265911977726,-74.18848456176755
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,40.6003239446019,-74.07177069948794
7,Queens,Astoria,Boro Zone,40.761491939310815,-73.91969421656458
8,Queens,Astoria Park,Boro Zone,40.778558067438325,-73.92308604147347
9,Queens,Auburndale,Boro Zone,40.751034246439524,-73.78794907079761
10,Queens,Baisley Park,Boro Zone,40.6789525649555,-73.79098661103701


In [0]:
zone_lookup.count()
zone_lookup.select("borough_name").distinct().show()

+-------------+
| borough_name|
+-------------+
|          EWR|
|          N/A|
|Staten Island|
|      Unknown|
|        Bronx|
|     Brooklyn|
|    Manhattan|
|       Queens|
+-------------+



##Joining geography onto the trips and generating the H3 cells

In [0]:
from pyspark.sql import functions as F, types as T

In [0]:
trips_raw = spark.table("workspace.bronze.trips_raw")
dim_zone = spark.table("workspace.silver.dim_zone")

pickup_zone = dim_zone.select(
    F.col("loc_id").alias("pu_loc_id"),
    F.col("centroid_lat").alias("pickup_lat"),
    F.col("centroid_lon").alias("pickup_lon"),
    F.col("borough_name").alias("pickup_borough"),
)
dropoff_zone = dim_zone.select(
    F.col("loc_id").alias("do_loc_id"),
    F.col("centroid_lat").alias("dropoff_lat"),
    F.col("centroid_lon").alias("dropoff_lon"),
    F.col("borough_name").alias("dropoff_borough"),
)

trips_geo = (
    trips_raw
    .join(pickup_zone, trips_raw.origin_loc_id == pickup_zone.pu_loc_id, "left")
    .join(dropoff_zone, trips_raw.dest_loc_id == dropoff_zone.do_loc_id, "left")
)

In [0]:
%sql
SELECT * FROM silver.dim_zone

loc_id,borough_name,zone_name,service_zone,centroid_lat,centroid_lon
1,EWR,Newark Airport,EWR,40.69183115473482,-74.17400008189793
2,Queens,Jamaica Bay,Boro Zone,40.61674466444515,-73.83129879543229
3,Bronx,Allerton/Pelham Gardens,Boro Zone,40.86447314470219,-73.84742222112844
4,Manhattan,Alphabet City,Yellow Zone,40.723751609058645,-73.97696808881015
5,Staten Island,Arden Heights,Boro Zone,40.55265911977726,-74.18848456176755
6,Staten Island,Arrochar/Fort Wadsworth,Boro Zone,40.6003239446019,-74.07177069948794
7,Queens,Astoria,Boro Zone,40.761491939310815,-73.91969421656458
8,Queens,Astoria Park,Boro Zone,40.778558067438325,-73.92308604147347
9,Queens,Auburndale,Boro Zone,40.751034246439524,-73.78794907079761
10,Queens,Baisley Park,Boro Zone,40.6789525649555,-73.79098661103701


In [0]:
dim_zone = spark.table("silver.dim_zone")


dim_zone.groupBy("loc_id").count().filter("count > 1").show()

+------+-----+
|loc_id|count|
+------+-----+
|    56|    2|
|   103|    3|
+------+-----+



This will cause the data exploision bug: Duplicating trip records and corrupting downstream aggregations.

In [0]:
from pyspark.sql import functions as F, types as T

In [0]:
dim_zone.filter(F.col("loc_id").isin(56, 103)).show()

+------+------------+--------------------+------------+------------------+------------------+
|loc_id|borough_name|           zone_name|service_zone|      centroid_lat|      centroid_lon|
+------+------------+--------------------+------------+------------------+------------------+
|    56|      Queens|              Corona|   Boro Zone| 40.75181865267571|-73.85358160692643|
|   103|   Manhattan|Governor's Island...| Yellow Zone|  40.6887838475454|-74.01907259018242|
|    56|      Queens|              Corona|   Boro Zone|40.741406071464866|-73.85884529716004|
|   103|   Manhattan|Governor's Island...| Yellow Zone| 40.69876831686451|-74.04077059735947|
|   103|   Manhattan|Governor's Island...| Yellow Zone| 40.68985976886141|-74.04528814672521|
+------+------------+--------------------+------------+------------------+------------------+



In [0]:
#Dropping the duplicates
dim_zone_unique = dim_zone.dropDuplicates(["loc_id"])
dim_zone_unique.write.mode("overwrite").saveAsTable("silver.dim_zone")
dim_zone = spark.read.table("silver.dim_zone")
dim_zone.groupBy("loc_id").count().filter("count > 1").show()

+------+-----+
|loc_id|count|
+------+-----+
+------+-----+



Deduplication is complete

In [0]:
H3_RESOLUTION = 8

trips_clean = (
    trips_geo
    .withColumn("pickup_h3", F.expr(f"h3_longlatash3string(pickup_lon, pickup_lat, {H3_RESOLUTION})"))
    .withColumn("dropoff_h3", F.expr(f"h3_longlatash3string(dropoff_lon, dropoff_lat, {H3_RESOLUTION})"))
    .withColumn(
        "trip_duration_min",
        (F.unix_timestamp("dropoff_timestamp") - F.unix_timestamp("pickup_timestamp")) / 60
    )
    .withColumn(
        "avg_speed_mph",
        F.when(F.col("trip_duration_min") > 0,
               F.col("distance_miles") / (F.col("trip_duration_min") / 60))
    )
    .withColumn(
        "is_valid_trip",
        (F.col("trip_duration_min") > 0) & (F.col("distance_miles") > 0)
        & F.col("pickup_lat").isNotNull() & F.col("dropoff_lat").isNotNull()
    )
)

**Something I learned:** Databrick's native funtion for taking geospatial data [h3_longlatash3string] works in the order of (longitude, latitude). But the h3's function of get_h3_cell takes inputs as (latitude, longitude). So, gotta be carefull with working with the two functions

h3.latlng_to_cell(lat, lon, res) - H3 library

h3_longlatash3string(lon, lat, res) - Python's inbuild function to take geospatial inputs

In [0]:
trips_clean.orderBy("provider_code").show()


+-------------+-------------------+-------------------+-----------+--------------+-------------+-------------------+-------------+-----------+----------------------+---------+--------------+-----------+------------------+----------+-----------------------+------------+-------------------+-----------+---------------------+--------------------+------------+---------+------------------+------------------+--------------+---------+------------------+------------------+---------------+---------------+---------------+------------------+------------------+-------------+
|provider_code|   pickup_timestamp|  dropoff_timestamp|rider_count|distance_miles|rate_class_id|offline_record_flag|origin_loc_id|dest_loc_id|fare_settlement_method|base_fare|surcharge_misc|transit_tax|driver_tip_payment|toll_total|service_improvement_fee|charge_total|zone_congestion_fee|Airport_fee|congestion_relief_fee|         source_file|source_month|pu_loc_id|        pickup_lat|        pickup_lon|pickup_borough|do_loc_id| 

In [0]:
%sql
DROP TABLE silver.dim_zone;

In [0]:
%sql
DROP TABLE silver.trips_clean;

In [0]:
if spark.sql("SHOW TABLES").filter("tableName = silver.trips_clean").count == 0:
    spark.sql("CREATE TABLE trips_clean")
    trips_clean.write.mode("overwrite").saveAsTable("workspace.silver.trips_clean")
else:
    trips_clean.write.mode("overwrite").saveAsTable("workspace.silver.trips_clean")

In [0]:
dim_zone = zone_lookup.join(zone_geo_spark, on="loc_id", how="left").dropDuplicates(["loc_id"])
dim_zone.write.mode("overwrite").saveAsTable("workspace.silver.dim_zone")